In [ ]:
!date

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'

donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Fail1', 'Delayed', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
ips_clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/label_transfer/snm3c_ips_subtype.csv'
ips_clusters = pd.read_csv(ips_clusters_path, sep='\t', index_col=0)
ips_clusters.shape

In [ ]:
ips_clusters.head()

In [ ]:
adata_path = f'{projdir}/h5ad/integrate_m3c/merged.processed.harmony.h5ad'
adata = sc.read_h5ad(adata_path)

In [ ]:
adata

In [ ]:
adata.obs.index = [x.replace('-0', '').replace('-1', '') for x in adata.obs.index]

In [ ]:
adata.obs['ips_subtype'] = ips_clusters.reindex(adata.obs.index)['ips_subtype']

In [ ]:
ips_adata = adata[~adata.obs['ips_subtype'].isna()]
ips_adata

In [ ]:
sc.pp.highly_variable_genes(ips_adata, n_top_genes=20000)

In [ ]:
sc.tl.pca(ips_adata, n_comps=50)

In [ ]:
sc.pl.pca(ips_adata, color=['ips_subtype', 'line'], components=['1,2','3,4'])

In [ ]:
sc.pp.neighbors(ips_adata, random_state=0)
sc.tl.umap(ips_adata, random_state=0)

In [ ]:
sc.pl.umap(ips_adata, color=['ips_subtype', 'line'])

In [ ]:
ips_adata.write_h5ad(f'{projdir}/h5ad/chrom100k/ips_subtype.processed.h5ad')

### write to df

In [ ]:
umap_df = pd.DataFrame(index=ips_adata.obs.index)
umap_df['umap0'] = ips_adata.obsm['X_umap'][:,0]
umap_df['umap1'] = ips_adata.obsm['X_umap'][:,1]
umap_df['ips_subtype'] = ips_adata.obs['ips_subtype']
umap_df['line'] = ips_adata.obs['line']
umap_df['time'] = ips_adata.obs['time']

In [ ]:
umap_df

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure5'
umap_df.to_csv(f'{outdir}/ips_subtype_mcg_umap_df.csv', sep='\t')

In [ ]:
!date